[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimyortega55-collab/motor-OCR/blob/main/entrenamiento/colab_prueba_humo.ipynb)

# Prueba de humo: fine-tuning de pix2tex en Colab

Este notebook solo confirma que el bucle de fine-tuning de
`entrenamiento/entrenar.py` corre de punta a punta en Colab con GPU y guarda un
checkpoint. Genera un dataset minusculo (300 formulas de entrenamiento, 40 de
validacion) y corre **1 sola epoca**: no produce un modelo con calidad real ni
lo pretende. Cuando esto pase, segui con
[`colab_finetuning_real.ipynb`](https://colab.research.google.com/github/rimyortega55-collab/motor-OCR/blob/main/entrenamiento/colab_finetuning_real.ipynb),
que usa un dataset grande y guarda los checkpoints en Drive.

Antes de correr: **Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion
-> GPU** (T4 esta en el tier gratis).

## 0. (Solo si vas a usar VS Code) Abrir un tunel SSH hacia esta VM

Corre esta celda **aqui, en el navegador de Colab**, una sola vez. Te pide una
clave temporal (no la reutilices en otro lado) y al final imprime un bloque para
pegar en tu `~/.ssh/config`. Con eso, VS Code (extension **Remote - SSH**) se
conecta a esta misma maquina con GPU y desde ahi corres el resto de las celdas
como comandos de terminal, o abris este mismo notebook con el kernel remoto.

Si te quedas en el navegador de Colab, saltatela.

In [ ]:
!pip install -q colab-ssh --upgrade
from colab_ssh import launch_ssh_cloudflared
from getpass import getpass

clave_temporal = getpass("Clave temporal para la sesion SSH (no la reutilices): ")
launch_ssh_cloudflared(password=clave_temporal)

## 1. Confirmar que hay GPU asignada

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No hay GPU asignada. Entorno de ejecucion -> Cambiar tipo de entorno de "
    "ejecucion -> GPU, y volve a correr desde esta celda."
)
gpu = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | {gpu.name} | {gpu.total_memory / 1024**3:.1f} GB")

## 2. Clonar el repo

In [ ]:
import os

REPO_URL = "https://github.com/rimyortega55-collab/motor-OCR.git"
REPO_DIR = "/content/motor-OCR"

if os.path.isdir(REPO_DIR):
    # Esta celda se vuelve a correr seguido (Colab desconecta el entorno y hay
    # que rehacer todo); `git clone` falla si el directorio ya existe.
    !git -C /content/motor-OCR pull --ff-only
else:
    !git clone {REPO_URL} /content/motor-OCR

%cd /content/motor-OCR
!git log --oneline -1

## 3. Instalar dependencias

Solo lo que necesita el entrenamiento (`pix2tex` trae torch, timm,
x-transformers, etc.). No se instala el resto del proyecto (easyocr, doctr,
transformers...) porque no hace falta para esta prueba.

`imagesize` y `python-Levenshtein` los importa `pix2tex` (en `dataset/dataset.py`
y en `eval.py`) pero solo los declara en su extra `[train]`. Se instalan
explicitamente en vez de usar `pix2tex[train]` porque ese extra arrastra
`torchtext`, que fuerza un downgrade de `torch`; `entrenamiento/_compat.py` ya
sustituye lo unico que pix2tex usa de `torchtext` (`bleu_score`).

La distribucion de LaTeX es para `pdflatex`, que es lo que renderiza cada
formula del dataset. Tarda varios minutos: es la parte lenta de esta prueba.

In [ ]:
!pip install -q pix2tex wandb python-Levenshtein pymupdf imagesize
!apt-get -qq update
!apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended texlive-latex-recommended

In [ ]:
import shutil

import torch

# `pip install pix2tex` puede arrastrar una version de torch distinta a la que
# Colab trae preinstalada y dejar la GPU sin usar.
print("torch:", torch.__version__, "| cuda disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Se perdio el acceso a la GPU al instalar dependencias. Entorno de ejecucion "
    "-> Reiniciar entorno de ejecucion, y volve a correr desde la celda del clon "
    "(no hace falta reinstalar: los paquetes sobreviven al reinicio)."
)
assert shutil.which("pdflatex"), (
    "pdflatex no quedo en el PATH: revisa la salida de apt-get de la celda anterior."
)
print("pdflatex:", shutil.which("pdflatex"))

## 4. Generar el dataset de la prueba

`entrenamiento/dataset_validacion/` **no viene en el clon**: los datasets
generados estan en `.gitignore` -se regeneran, y ademas sus `.pkl` guardan las
rutas de las imagenes tal como existian en la maquina que los creo, asi que
traerlos no serviria. Se genera uno equivalente aca mismo, chico a proposito.

In [ ]:
DATASET_DIR = "entrenamiento/dataset_humo"

!python entrenamiento/generar_dataset_sintetico.py --n-train 300 --n-val 40 --out {DATASET_DIR}

In [ ]:
from pathlib import Path

# generar_dataset_sintetico.py no falla cuando pdflatex no puede renderizar una
# formula: deja la linea vacia y sigue. Si LaTeX quedo mal instalado el dataset
# sale vacio y el entrenamiento arrancaria contra la nada.
for split in ("train", "val"):
    dir_split = Path(DATASET_DIR) / split
    lineas = (dir_split / "formulas.txt").read_text(encoding="utf-8").splitlines()
    renderizadas = sum(1 for linea in lineas if linea.strip())
    print(f"{split}: {renderizadas}/{len(lineas)} formulas renderizadas")
    assert renderizadas > 0, (
        f"El split '{split}' quedo vacio: pdflatex no renderizo ni una formula. "
        "Revisa la salida de la celda de instalacion de LaTeX."
    )

## 5. Empaquetar el dataset y adaptar la config a las rutas de Colab

`entrenamiento/config_validacion.yaml` apunta a `.venv/Lib/site-packages/pix2tex/...`
porque se escribio para el venv local en Windows. En Colab, `pip install pix2tex`
deja esos mismos archivos (checkpoint pre-entrenado + tokenizer) en otra ruta,
asi que se resuelve en vivo y se genera una copia de la config con las rutas
correctas.

El empaquetado usa el tokenizer del checkpoint pre-entrenado, no uno nuevo: con
un vocabulario distinto los pesos pre-entrenados no significarian nada.

In [ ]:
import os

import pix2tex
import yaml
from pix2tex.model.checkpoints.get_latest_checkpoint import download_checkpoints

base_pix2tex = os.path.dirname(pix2tex.__file__)
CHECKPOINT = os.path.join(base_pix2tex, "model", "checkpoints", "weights.pth")
TOKENIZER = os.path.join(base_pix2tex, "model", "dataset", "tokenizer.json")

if not os.path.exists(CHECKPOINT):
    # El wheel de PyPI trae el tokenizer pero no los pesos (~100 MB).
    download_checkpoints()

assert os.path.exists(CHECKPOINT), f"No se encontro el checkpoint pre-entrenado en {CHECKPOINT}"
assert os.path.exists(TOKENIZER), f"No se encontro el tokenizer en {TOKENIZER}"

!python -m pix2tex.dataset.dataset -i {DATASET_DIR}/train/imagenes -e {DATASET_DIR}/train/formulas.txt -t {TOKENIZER} -o {DATASET_DIR}/train.pkl
!python -m pix2tex.dataset.dataset -i {DATASET_DIR}/val/imagenes -e {DATASET_DIR}/val/formulas.txt -t {TOKENIZER} -o {DATASET_DIR}/val.pkl

with open("entrenamiento/config_validacion.yaml") as f:
    config = yaml.safe_load(f)

config["load_chkpt"] = CHECKPOINT
config["tokenizer"] = TOKENIZER
config["data"] = f"{DATASET_DIR}/train.pkl"
config["valdata"] = f"{DATASET_DIR}/val.pkl"

with open("entrenamiento/config_humo_colab.yaml", "w") as f:
    yaml.safe_dump(config, f)

print("load_chkpt:", config["load_chkpt"])
print("data:", config["data"])

## 6. Correr la prueba de humo

`config_validacion.yaml` trae `debug: true`, pero eso solo no alcanza para
desactivar wandb: `parse_args` (en `pix2tex/utils/utils.py`) sobreescribe
`args.debug` con el default del propio CLI (`False`) cuando no se pasa `--debug`
en la linea de comandos, asi que `args.wandb` termina en `True` igual y el
arranque revienta en `wandb.util.generate_id()`. Por eso se pasa `--debug`
explicito: no hace falta iniciar sesion en wandb para esta prueba.

In [ ]:
!python entrenamiento/entrenar.py --config entrenamiento/config_humo_colab.yaml --debug

## 7. Confirmar que se guardo un checkpoint

In [ ]:
!ls -la entrenamiento/checkpoints_validacion/pix2tex_val/

Si esta celda lista al menos un archivo `.pth`, el bucle de fine-tuning corrio
y guardo el modelo: la prueba de humo paso.

El checkpoint queda en el disco local de Colab y se pierde al desconectarse; no
importa, no sirve para nada mas que confirmar que el flujo funciona. El siguiente
paso es
[`colab_finetuning_real.ipynb`](https://colab.research.google.com/github/rimyortega55-collab/motor-OCR/blob/main/entrenamiento/colab_finetuning_real.ipynb),
que si guarda los checkpoints en Drive.